# Plant Detection using TensorFlow Object Detection API

This notebook demonstrates training and evaluation of an object detection model
for plant instance detection on the :contentReference[oaicite:0]{index=0} dataset.

We fine-tune a pre-trained SSD model:
- Model: SSD MobileNet V2 FPNLite (320×320)
- Framework: TensorFlow Object Detection API (TF 2.11)
- Task: Crop vs Weed detection

The notebook covers:
1. Environment setup
2. Dataset loading
3. Model configuration
4. Training and evaluation
5. Qualitative inference results

## Environment Setup

This experiment is designed for:
- Python 3.10
- TensorFlow 2.11
- CUDA-enabled GPU

On Kaggle:
Enable "Pin to original environment" to avoid version drift.

In [ ]:
!python -V

In [ ]:
# --- Core: minimal + compatible setup for TF 2.11 OD API ---

!pip install -q \
  numpy==1.22.4 \
  scipy==1.10.1 \
  protobuf==3.20.3 \
  tf_slim \
  pycocotools \
  lvis \
  Cython \
  contextlib2 \
  pillow \
  matplotlib \
  gin-config \
  tf-models-official==2.11.0 --no-deps

In [ ]:
!pip install \
  --upgrade \
  --force-reinstall \
  --no-cache-dir \
  --no-deps \
  git+https://github.com/frdiener/agri-vision-edge.git

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print("GPU ready:", gpus)

In [ ]:
from agri_vision_edge.third_party import setup_tensorflow_models
setup_tensorflow_models()

import object_detection
import google.protobuf

print("TF:", tf.__version__)
print("protobuf:", google.protobuf.__version__)

## Dataset

We use preprocessed TFRecord files derived from :contentReference[oaicite:1]{index=1}.

Additionally, raw images are used for qualitative evaluation.

In [ ]:
from pathlib import Path
import agri_vision_edge

dataset_dir = Path("/kaggle/input/datasets/freimutdiener/phenobench-tfrecord-dataset-for-tensorflow-od")
dataset_raw_dir = Path("/kaggle/input/datasets/freimutdiener/phenobench-raw-dataset-v1-1-0/PhenoBench")

label_map_path = dataset_dir / "label_map.pbtxt"
train_record = dataset_dir / "train.record"
val_record = dataset_dir / "val.record"

test_imgs = list((dataset_raw_dir / "test" / "images").glob("*.png"))

print(f"{len(test_imgs)} test images loaded")

## Model Configuration

We fine-tune a pre-trained SSD model:
- Backbone: MobileNetV2
- Feature extractor: FPNLite
- Input size: 320×320

We adapt:
- number of classes
- learning rate schedule
- dataset paths

In [ ]:
from agri_vision_edge.tfod import configure_ssd_pipeline

MODEL_DIR = Path("/kaggle/input/models/freimutdiener/ssd-mobilenet-v2-fpnlite-320x320-coco17-tpu-8/tensorflow2/default/1/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8")

PIPELINE_CONFIG = Path("/kaggle/working/pipeline.config")

configure_ssd_pipeline(
    config_path=MODEL_DIR / "pipeline.config",
    output_path=PIPELINE_CONFIG,
    train_record=train_record,
    val_record=val_record,
    label_map=label_map_path,
    checkpoint_path=MODEL_DIR / "checkpoint" / "ckpt-0",
    num_classes=2,
    batch_size=4,  # should really be more for stable gradient
    learning_rate_base=0.001,
    warmup_learning_rate=0.0002,
    num_steps=10_000,
    warmup_steps=500,

)

## Training

We train for 10,000 steps using fine-tuning from COCO weights.

In [ ]:
from agri_vision_edge.tfod import launch_training

training_process = launch_training(
    pipeline_config_path=PIPELINE_CONFIG,
    model_dir="/kaggle/working/training",
    checkpoint_every_n=250,
    checkpoint_max_to_keep=200,
    log_file="train.log",
    background=False,
)

## Training Metrics and Learning Curves

The TensorFlow Object Detection API writes training metrics
as TensorBoard event files during optimization.

We parse these logs to generate publication-quality plots
for:

- total training loss
- classification loss
- localization loss
- learning rate schedule
- training throughput

These figures can later be exported directly for inclusion
in reports, presentations, or academic theses.

In [ ]:
from agri_vision_edge.evaluation import (
    load_event_scalars,
    plot_loss_curves,
    plot_learning_rate,
    plot_steps_per_second,
    available_tags,
)

train_df = load_event_scalars(
    "/kaggle/working/training/train"
)

print("Available TensorBoard tags:")
for tag in available_tags(train_df):
    print("-", tag)

In [ ]:
plot_loss_curves(
    train_df,
    smoothing=0.6,
    save_path="training_loss_curves.pdf",
)

In [ ]:
plot_learning_rate(
    train_df,
    save_path="learning_rate_schedule.pdf",
)

In [ ]:
plot_steps_per_second(
    train_df,
    smoothing=0.5,
    save_path="training_throughput.pdf",
)

The generated figures are also exported as PDF files:

- `training_loss_curves.pdf`
- `learning_rate_schedule.pdf`
- `training_throughput.pdf`

These vector graphics are suitable for direct inclusion
in scientific publications and LaTeX-based theses.

In [ ]:
# from agri_vision_edge.tfod import launch_eval

# launch_eval(
#     pipeline_config_path=PIPELINE_CONFIG,
#     checkpoint_dir="/kaggle/working/training",
#     model_dir="/kaggle/working/eval_output",
# )


## Validation and Checkpoint Selection

Instead of evaluating only the final checkpoint,
we evaluate all saved checkpoints throughout training.

This enables:

- training progression analysis
- mAP evolution tracking
- best checkpoint selection
- overfitting detection

In [ ]:
from agri_vision_edge.evaluation import evaluate_checkpoints


metrics_df = evaluate_checkpoints(
    pipeline_config_path=PIPELINE_CONFIG,
    checkpoint_dir="/kaggle/working/training",
    output_dir="/kaggle/working/eval_checkpoints",
)

metrics_df

## Validation Metrics

We evaluate all saved checkpoints using standard
COCO-style object detection metrics.

This enables analysis of:

- convergence behavior
- validation performance evolution
- checkpoint selection
- potential overfitting

In [ ]:
metrics_df.columns

In [ ]:
metrics_df[
    [
        "step",
        "DetectionBoxes_Precision/mAP",
        "DetectionBoxes_Precision/mAP@.50IOU",
        "DetectionBoxes_Precision/mAP@.75IOU",
        "DetectionBoxes_Recall/AR@100",
    ]
]

In [ ]:
metrics_df

In [ ]:
from agri_vision_edge.evaluation import plot_checkpoint_metrics

evaluation.plot_checkpoint_metrics(
    metrics_df,
)

In [ ]:
from agri_vision_edge.evaluation import find_best_checkpoint

best_checkpoint = find_best_checkpoint(
    metrics_df,
    metric="DetectionBoxes_Precision/mAP",
)

best_checkpoint

## Model Export

We export the trained model in two formats:

- a standard TensorFlow SavedModel for generic TensorFlow
  inference workflows
- a TensorFlow Lite–compatible export graph for quantization
  and embedded deployment

The TensorFlow Lite export path uses TensorFlow Object
Detection's dedicated TFLite exporter, which rewrites the
graph for improved compatibility with:

- TensorFlow Lite conversion
- post-training quantization
- embedded accelerators
- NPU delegates such as TIM-VX / Teflon

This export path typically reduces dynamic TensorFlow ops
and improves deployment compatibility on edge devices.

In [ ]:
from agri_vision_edge.tfod import export_saved_model

export_saved_model(
    pipeline_config_path=PIPELINE_CONFIG,
    trained_checkpoint_dir="/kaggle/working/training",
    checkpoint_path=best_checkpoint["checkpoint"],
    output_directory="/kaggle/working/exported_model",
)

In [ ]:
from agri_vision_edge.tfod import export_tflite_graph

export_tflite_graph(
    pipeline_config_path=PIPELINE_CONFIG,
    trained_checkpoint_dir="/kaggle/working/training",
    checkpoint_path=best_checkpoint["checkpoint"],
    output_directory="/kaggle/working/tflite_export",
)

## Remove Checkpoints to prevent storage bloat.

In [ ]:
!rm -rf {training,eval_checkpoints}

## Qualitative Evaluation

We visualize predictions on unseen test images.

In [ ]:
from IPython.display import display
from agri_vision_edge.tfod.inference import (
    load_saved_model,
    load_label_map,
    detect_image,
)

detect_fn = load_saved_model(
    "/kaggle/working/exported_model/saved_model"
)

category_index = load_label_map(label_map_path)

for image_path in test_imgs[:10]:
    vis, _ = detect_image(
        detect_fn=detect_fn,
        image_path=image_path,
        category_index=category_index,
        image_size=320,
        score_threshold=0.2,
        max_boxes=20,
    )
    display(vis)

## Discussion

The model demonstrates:

- Successful localization of plant instances
- Overlapping detections reduced via NMS
- Sensitivity to small objects (PhenoBench-specific challenge)

Limitations:
- Performance depends strongly on resolution (320×320)
- Dense scenes produce multiple candidate detections
- Further improvements possible via:
  - anchor tuning
  - longer training
  - quantization-aware training (QAT)

Future work:
- INT8 deployment via TFLite
- Real-time inference optimization